In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import pandas as pd
import re
from sklearn.preprocessing import MultiLabelBinarizer
from collections import Counter

def clean_uniprot_data(csv_path, max_seq_length=1000, min_label_freq=50):
    """
    清洗 UniProt 数据并提取多标签
    """
    print("开始加载并清洗数据...")
    df = pd.read_csv(csv_path)
    
    # 1. 剔除缺失值
    df = df.dropna(subset=['Sequence', 'Gene Ontology (GO)']).copy()
    
    # 2. 限制序列长度 (预留一点空间给特殊 token，所以设为 1000 而不是 1024)
    df['seq_length'] = df['Sequence'].apply(len)
    df = df[df['seq_length'] <= max_seq_length].copy()
    
    # 3. 使用正则表达式提取 GO ID
    # 匹配模式为 "GO:" 加上 7 位数字
    def extract_go_terms(go_string):
        return re.findall(r'GO:\d{7}', str(go_string))
    
    df['go_terms'] = df['Gene Ontology (GO)'].apply(extract_go_terms)
    
    # 4. 统计标签频率并过滤低频长尾标签
    all_go_terms = [term for terms in df['go_terms'] for term in terms]
    term_counts = Counter(all_go_terms)
    
    # 仅保留出现次数 >= min_label_freq 的标签
    valid_terms = {term for term, count in term_counts.items() if count >= min_label_freq}
    
    # 过滤每条数据的标签列表，剔除低频标签
    df['filtered_go_terms'] = df['go_terms'].apply(lambda terms: [t for t in terms if t in valid_terms])
    
    # 剔除过滤后标签为空的蛋白质序列
    df = df[df['filtered_go_terms'].map(len) > 0].copy()
    
    print(f"清洗完成！有效样本数: {len(df)}")
    print(f"保留的唯一 GO 标签数量: {len(valid_terms)}")
    
    return df

# 运行清洗逻辑 (假设你的文件名为 uniprot_proteins.csv)
raw_csv_path = "/content/drive/MyDrive/Protein_Project/uniprot_proteins.csv"
df_cleaned = clean_uniprot_data(raw_csv_path, max_seq_length=1000, min_label_freq=50)

# ===== 【新增代码】将清洗后的结果保存为新的 CSV 文件 =====
cleaned_csv_path = "/content/drive/MyDrive/Protein_Project/uniprot_proteins_cleaned.csv"
df_cleaned.to_csv(cleaned_csv_path, index=False)
print(f"清洗后的数据已成功保存至: {cleaned_csv_path}")

开始加载并清洗数据...
清洗完成！有效样本数: 16931
保留的唯一 GO 标签数量: 685
清洗后的数据已成功保存至: /content/drive/MyDrive/Protein_Project/uniprot_proteins_cleaned.csv


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from torch.optim import AdamW
from tqdm import tqdm

# ==========================================
# 1. 路径与环境配置 (适配 Colab 挂载的云盘)
# ==========================================
# 指定你在 Google Drive 中的项目路径
PROJECT_DIR = "/content/drive/MyDrive/Protein_Project"

# 数据集路径
DATA_PATH = os.path.join(PROJECT_DIR, "uniprot_proteins_cleaned.csv")

# 模型路径：如果模型已上传云盘则用路径，否则直接用 Hugging Face 镜像
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com' # 防断联镜像
MODEL_CHECKPOINT = "facebook/esm2_t6_8M_UR50D" 
# 如果你传到了云盘，可以改为: MODEL_CHECKPOINT = os.path.join(PROJECT_DIR, "esm2_model")

# ==========================================
# 2. 数据加载类 (保持不变)
# ==========================================
class ProteinFunctionDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_length=1024):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            seq, padding='max_length', truncation=True, 
            max_length=self.max_length, return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.float32)
        return item

# ==========================================
# 3. 数据预处理与 DataLoader 封装
# ==========================================
def prepare_data(data_path, model_checkpoint, batch_size=8):
    print("正在加载云盘数据...")
    df_cleaned = pd.read_csv(data_path)
    
    # 假设你的 csv 中处理好的标签列叫 'filtered_go_terms'
    # 注意：从 csv 读取列表时可能会变成字符串，需要安全转换
    import ast
    df_cleaned['filtered_go_terms'] = df_cleaned['filtered_go_terms'].apply(ast.literal_eval)
    
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    mlb = MultiLabelBinarizer()
    y_encoded = mlb.fit_transform(df_cleaned['filtered_go_terms'])
    num_labels = len(mlb.classes_)
    
    train_seqs = df_cleaned['Sequence'].tolist()
    train_labels = y_encoded
    
    train_dataset = ProteinFunctionDataset(train_seqs, train_labels, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    return train_loader, num_labels

# ==========================================
# 4. 主训练循环 (强制调用 Colab GPU)
# ==========================================
if __name__ == "__main__":
    # 确认是否成功分配到了 Colab 的 GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"当前计算设备: {device}")
    if device.type == "cpu":
        print("警告：未检测到 GPU，请检查 Colab 的运行时设置！")

    # 获取 DataLoader
    train_loader, num_labels = prepare_data(DATA_PATH, MODEL_CHECKPOINT, batch_size=8)
    
    # 加载带有分类头的 ESM-2 模型
    print("正在加载 ESM-2 模型...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, 
        num_labels=num_labels, 
        problem_type="multi_label_classification"
    )
    model = model.to(device)
    
    optimizer = AdamW(model.parameters(), lr=5e-5)
    epochs = 3
    
    print("🚀 开始训练...")
    for epoch in range(epochs):
        model.train() 
        total_train_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_train_loss += loss.item()
            
            loss.backward()
            optimizer.step()
            
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        avg_train_loss = total_train_loss / len(train_loader)
        print(f"\n--- Epoch {epoch+1} 结束 ---")
        print(f"平均训练损失: {avg_train_loss:.4f}\n")
        
        # 训练完成后，将模型权重保存回 Google Drive
        save_path = os.path.join(PROJECT_DIR, f"esm2_finetuned_epoch_{epoch+1}")
        model.save_pretrained(save_path)
        print(f"✅ 当前轮次模型已安全保存至云盘: {save_path}")

当前计算设备: cuda
正在加载云盘数据...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 开始训练...


Epoch 1/3:  14%|█▍        | 303/2117 [02:19<15:02,  2.01it/s, loss=0.0689]

In [ ]:
import torch
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ==========================================
# 1. 准备验证集 (Validation Set)
# ==========================================
# 读取你之前清洗好的数据
df_cleaned = pd.read_csv(DATA_PATH)
import ast
df_cleaned['filtered_go_terms'] = df_cleaned['filtered_go_terms'].apply(ast.literal_eval)

# 重新初始化 Binarizer 获取多热编码
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(df_cleaned['filtered_go_terms'])

# 划分 20% 的数据作为验证集 (不参与训练，用于测试模型真实的泛化能力)
_, val_seqs, _, val_labels = train_test_split(
    df_cleaned['Sequence'].tolist(), 
    y_encoded, 
    test_size=0.2, 
    random_state=42 # 固定随机种子以保证结果可复现
)

# ==========================================
# 2. 加载训练好的模型
# ==========================================
# 我们选择最后一轮 (Epoch 3) 的模型来进行验证
SAVED_MODEL_PATH = os.path.join(PROJECT_DIR, "esm2_finetuned_epoch_3")
print(f"正在加载微调后的模型: {SAVED_MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT) # Tokenizer 用基础的即可
model_eval = AutoModelForSequenceClassification.from_pretrained(SAVED_MODEL_PATH)
model_eval = model_eval.to(device)
model_eval.eval() # 极其重要：将模型切换到评估模式 (关闭 Dropout 等)

# 封装验证集 DataLoader
val_dataset = ProteinFunctionDataset(val_seqs, val_labels, tokenizer, max_length=1000)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# ==========================================
# 3. 运行验证循环与指标计算
# ==========================================
all_preds = []
all_labels = []

print("开始在验证集上评估模型...")
with torch.no_grad(): # 验证阶段不需要计算梯度，极大节省显存和时间
    for batch in tqdm(val_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # 前向传播
        outputs = model_eval(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # 使用 Sigmoid 将输出映射到 0~1 的概率区间
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
        all_labels.append(labels.cpu().numpy())

# 将 batch 结果拼接成完整的矩阵
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

# ==========================================
# 4. 统计与打印最终指标
# ==========================================
# 将概率转化为 0/1 预测结果 (通常以 0.5 为阈值)
threshold = 0.5
binary_preds = (all_preds > threshold).astype(int)

# 计算各项 Macro 指标 (对所有标签求平均)
# 注意：如果有极低频标签在验证集中全是 0，会导致计算警告，这里使用 zero_division=0 忽略
macro_f1 = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
micro_f1 = f1_score(all_labels, binary_preds, average='micro', zero_division=0)
macro_auprc = average_precision_score(all_labels, all_preds, average='macro')

print("\n" + "="*40)
print("🎯 验证集评估结果 (Evaluation Metrics)")
print("="*40)
print(f"Macro F1-Score: {macro_f1:.4f}  (各类标签预测能力的平均水平)")
print(f"Micro F1-Score: {micro_f1:.4f}  (整体预测的准确度)")
print(f"Macro AUPRC:    {macro_auprc:.4f}  (极度不平衡数据下的核心指标，越高越好)")
print("="*40)